# 3.4 — Correlación entre variables

**Aspecto del rúbrica:** *Correlación entre variables*.

La **correlación** es un número entre **−1 y +1** que dice qué tanto se mueven
juntas dos variables:

- **+1**: cuando una sube, la otra siempre sube.
- **−1**: cuando una sube, la otra siempre baja.
- **0**: no se mueven juntas.

Como guía: menos de 0.3 (sin importar el signo) es una relación débil, de 0.3 a
0.5 moderada y más de 0.5 fuerte. Una correlación **no prueba causa**: dos cosas
pueden moverse juntas solo porque las dos dependen de una tercera, por ejemplo
la época del año.

Esta libreta responde cuatro preguntas:

1. ¿Qué tanto se relacionan los casos de IRA con el clima y el aire, con los datos originales?
2. ¿Esas relaciones siguen ahí al quitar la época del año?
3. ¿El clima de las semanas anteriores se relaciona con los casos?
4. ¿Qué variables de clima y aire se repiten entre sí? (sirve para 3.5)

## Resumen

- **Con los datos originales, la temperatura y el PM10 parecen muy ligados a los casos**
  (−0.78 y +0.49): más casos cuando hace frío y hay más polvo.
- **Pero es la época del año.** Comparando cada semana con lo normal para su
  mes, esas correlaciones caen a casi 0 (−0.02 y +0.12). Confirma lo que se vio
  en 3.3.
- **Solo quedan dos señales débiles:** más ozono de lo normal va con un poco más
  de casos (+0.21), pero baja a +0.12 sin 2016, el año del ozono dudoso (3.2);
  y el frío fuera de lo normal **3 a 5 semanas antes** va con un poco más de
  casos (alrededor de −0.14).
- **Varias variables de clima y aire se repiten entre sí** (temperatura, humedad
  y PM10, con correlaciones de 0.5 a 0.7): conviene resumirlas en 3.5.

## Cómo se hace

- **Datos originales vs. sin la época:** en toda la libreta se comparan dos
  versiones de los mismos datos. *Datos originales* son los valores como se
  midieron (por ejemplo, 6,356 casos y 28.2 °C en una semana de enero). *Sin la
  época* es cada valor menos lo normal para su mes (+191 casos y +3.8 °C: esa
  semana tuvo 191 casos y 3.8 °C más que un enero normal). La diferencia entre
  las dos versiones es lo que aporta la época del año.
- **Dos formas de medir:** *Pearson* mide si los puntos forman una línea recta;
  *Spearman* solo mira el orden (si cuando una sube la otra también), así que no
  le afectan tanto los valores extremos. Se usa **Spearman** como medida
  principal y Pearson solo para comparar.
- **Se deja fuera 2020:** los casos cayeron por la pandemia, no por el clima
  (`anio_atipico`, igual que en 3.3). Quedan 260 semanas (2015–2019).
- **Solo valores medidos:** se usa el panel **sin rellenar** (los huecos de aire
  quedan vacíos), para no medir relaciones con datos estimados por 3.1.
- **Se dejan fuera** las olas de calor (casi siempre valen 0; ver 3.2 y 3.3) y
  la marginación (`IM`), que sube en línea recta y solo mediría la tendencia
  del tiempo.

| Entrada | Salida |
|---------|--------|
| `data/processed/panel_eda_semanal.parquet` o `panel_eda.parquet` (según `GRANULARIDAD`), del paso 7 de `3.0` | Figuras `reports/figures/corr_*_{GRANULARIDAD}.png` |

## Configuración e imports

Se carga el panel y se calcula, para cada variable, **cuánto se alejó cada
semana de lo normal para su mes** (el promedio de ese mes en 2015–2019). Es la
misma idea de 3.3: así se puede comparar una semana de enero solo contra otros
eneros.

In [ ]:
from pathlib import Path  # Path: manejar rutas de carpetas y archivos
import sys  # sys: cambiar dónde busca módulos Python

# Este notebook vive en notebooks/03_eda/: subimos dos carpetas hasta la raíz del proyecto
ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt  # gráficas
import numpy as np  # cálculos numéricos
import pandas as pd  # tablas
import seaborn as sns  # mapas de calor

from sonora_health_climate_eda.config import FIGURES_DIR, PROCESSED_DATA_DIR

# "semanal" = análisis principal; "mensual" = comparación
GRANULARIDAD = "semanal"

# Nombre del archivo y de las columnas según la granularidad (igual que en 3.2 y 3.3)
_CFG = {
    "semanal": {"archivo": "panel_eda_semanal.parquet", "fecha": "fecha_semana",
                "y": "casos_ira_total", "tmax": "tmax_media_sem"},
    "mensual": {"archivo": "panel_eda.parquet", "fecha": "fecha_mes",
                "y": "casos_ira_por_semana", "tmax": "tmax_media_mes"},
}[GRANULARIDAD]
FECHA, Y, TMAX = _CFG["fecha"], _CFG["y"], _CFG["tmax"]

# Variables de clima y aire que se comparan con los casos
VARIABLES = [TMAX, "humedad_rel_media", "severidad_sequia", "pm10_ugm3", "o3_max8h_ppb"]
TODAS = [Y] + VARIABLES
# Nombres cortos y claros para tablas y gráficas
NOMBRE = {Y: "Casos de IRA", TMAX: "Temperatura máx.", "humedad_rel_media": "Humedad",
          "severidad_sequia": "Sequía", "pm10_ugm3": "PM10", "o3_max8h_ppb": "Ozono"}

# Lee el panel. Si no existe, se detiene y dice qué notebook hay que correr.
ruta = PROCESSED_DATA_DIR / _CFG["archivo"]
if not ruta.exists():
    raise FileNotFoundError(f"Falta {ruta.name}: completar el paso 7 de 3.0-integracion-espacial.ipynb")
panel = pd.read_parquet(ruta).sort_values(FECHA).reset_index(drop=True)

# Se deja fuera 2020 (pandemia) y se agrega el mes del año (1 = enero ... 12 = diciembre)
base = panel[~panel["anio_atipico"].astype(bool)].reset_index(drop=True)
base["mes"] = base[FECHA].dt.month

# "Sin la época": cada valor menos el promedio de su mes del año.
# Ej.: si una semana de enero tuvo 6,500 casos y lo normal en enero es 6,200, queda +300.
# transform("mean") calcula el promedio de cada mes y lo repite en cada fila de ese mes.
sin_epoca = base[TODAS] - base.groupby("mes")[TODAS].transform("mean")

print(f"Panel {GRANULARIDAD}: {len(base)} periodos (2015-2019).")
print("Periodos con dato medido por variable:",
      {NOMBRE[c]: int(n) for c, n in base[TODAS].notna().sum().items()})

## 1. Correlación con los datos originales

Dos mapas de calor con las mismas variables: Pearson a la izquierda y Spearman a
la derecha. **Rojo** = se mueven juntas (correlación positiva), **azul** = una
sube cuando la otra baja (negativa), **blanco** = no se relacionan. Lo que más
importa es la **primera fila**: cómo se relaciona cada variable con los casos.

**Resultado:** la temperatura (−0.78 con Spearman) y el PM10 (+0.49) son las
más ligadas a los casos: más casos cuando hace frío y hay más polvo. La sequía y
el ozono apenas se relacionan (−0.18) y la humedad nada (0.01). Pearson y
Spearman dan casi lo mismo, así que no hay relaciones "curvas" escondidas.

In [ ]:
# .corr() calcula la correlación de cada pareja de columnas. Usa solo los periodos
# en los que las dos variables tienen dato (los huecos del aire no estorban).
corr_pearson = base[TODAS].corr(method="pearson").rename(index=NOMBRE, columns=NOMBRE)
corr_spearman = base[TODAS].corr(method="spearman").rename(index=NOMBRE, columns=NOMBRE)

# Dos mapas de calor lado a lado. annot=True escribe el número en cada cuadro;
# center=0 deja el blanco en 0, y vmin/vmax fijan la escala de -1 a +1.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (titulo, corr) in zip(axes, [("Pearson", corr_pearson), ("Spearman", corr_spearman)]):
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                square=True, cbar=False, ax=ax)
    ax.set_title(f"{titulo}, datos originales", loc="left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"corr_originales_{GRANULARIDAD}.png", dpi=150)
plt.show()

## 2. ¿Siguen ahí al quitar la época del año?

En 3.3 se vio que los casos, la temperatura y el PM10 tienen **su propio
calendario**: en invierno hace frío, hay más polvo y más casos, todo a la vez.
Eso basta para que salgan correlaciones fuertes aunque una cosa no cause la
otra. Aquí se repite la correlación con cada semana comparada con **lo normal
para su mes**.

**¿Podría ser casualidad?** El valor *p* dice qué tan seguido el puro azar daría
una correlación igual o más fuerte. Como semanas vecinas se parecen mucho, el
cálculo de siempre es demasiado optimista; por eso, como en 3.3, se revuelven
los casos **en bloques de 8 semanas** (2 meses en el panel mensual), 1,000
veces. Si *p* es menor a 0.05, es difícil que sea casualidad.

**Resultado:**

| Variable | Datos originales | Sin la época | *p* (sin la época) |
|----------|----------|--------------|--------------------|
| Temperatura máx. | −0.78 | −0.02 | 0.83 |
| PM10 | +0.49 | +0.12 | 0.18 |
| Sequía | −0.18 | +0.01 | 0.92 |
| Ozono | −0.18 | **+0.21** | **0.03** |
| Humedad | +0.01 | −0.10 | 0.19 |

Las relaciones fuertes **desaparecen**: eran la época del año. Solo el ozono
deja una señal débil (más ozono de lo normal, un poco más de casos), y es
frágil: sin 2016 baja a +0.12 (2016 tiene el ozono dudoso que marcó 3.2).

In [ ]:
# Para que el resultado sea el mismo cada vez que se corre
rng = np.random.default_rng(42)
TAM_BLOQUE = 8 if GRANULARIDAD == "semanal" else 2  # 8 semanas o 2 meses
# Lista de bloques de periodos seguidos: [0..7], [8..15], ...
bloques = [np.arange(i, min(i + TAM_BLOQUE, len(base))) for i in range(0, len(base), TAM_BLOQUE)]


def p_por_bloques(y: pd.Series, x: pd.Series, repeticiones: int = 1000) -> float:
    # Correlación real y, luego, cuántas veces el azar da una igual o más fuerte
    real = y.corr(x, method="spearman")
    veces = 0
    for _ in range(repeticiones):
        # Revuelve el ORDEN de los bloques (dentro de cada bloque no cambia nada)
        orden = np.concatenate([bloques[j] for j in rng.permutation(len(bloques))])
        y_revuelta = pd.Series(y.to_numpy()[orden], index=y.index)
        if abs(y_revuelta.corr(x, method="spearman")) >= abs(real):
            veces += 1
    return veces / repeticiones


# Una fila por variable: correlación con los casos con los datos originales, sin la época, y su p
tabla = pd.DataFrame({
    "datos originales": base[VARIABLES].corrwith(base[Y], method="spearman"),
    "sin la época": sin_epoca[VARIABLES].corrwith(sin_epoca[Y], method="spearman"),
    "p (sin la época)": [p_por_bloques(sin_epoca[Y], sin_epoca[v]) for v in VARIABLES],
}, index=VARIABLES).rename(index=NOMBRE)
display(tabla.round(2))

# Gráfica de barras: cada variable con sus dos correlaciones, una al lado de la otra
ax = tabla[["datos originales", "sin la época"]].plot.barh(figsize=(8, 4), color=["#9ec5f4", "#2a78d6"])
ax.axvline(0, color="#52514e", lw=1)  # línea en 0 = sin relación
ax.set_xlim(-1, 1)
ax.set_xlabel("correlación de Spearman con los casos de IRA")
ax.set_title("Al quitar la época del año, las relaciones casi desaparecen", loc="left")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"corr_originales_vs_sin_epoca_{GRANULARIDAD}.png", dpi=150)
plt.show()

# Revisión del ozono: ¿la señal depende de 2016, el año del ozono dudoso (3.2)?
sin_2016 = base[FECHA].dt.year != 2016
r_ozono = sin_epoca.loc[sin_2016, Y].corr(sin_epoca.loc[sin_2016, "o3_max8h_ppb"], method="spearman")
print(f"Ozono sin la época, sin 2016: {r_ozono:+.2f}")

## 3. ¿Y el clima de las semanas anteriores?

Una infección no aparece el mismo día del frío: puede tardar en notarse. Aquí se
compara la cantidad de casos de cada semana (sin la época) con cada variable
(sin la época) **0 a 8 semanas antes** (0 a 2 meses en el panel mensual).

**Cómo leerlo:** cada línea es una variable y cada punto, la correlación con
ese rezago. La franja gris marca la zona donde las correlaciones son tan chicas
(menos de ±0.1) que no dicen nada.

**Resultado:** la única que se sale de forma constante es la **temperatura**:
con 2 a 7 semanas de rezago queda alrededor de **−0.10 a −0.15**, es decir,
después de semanas más frías de lo normal hay un poco más casos. El valor más
fuerte es con 5 semanas (−0.15, *p* ≈ 0.05 con la prueba por bloques). Coincide
con 3.3, pero es una señal débil. La humedad sube a +0.15 con 5 semanas, pero con 0 semanas es negativa:
no es un patrón claro. El ozono queda entre +0.10 y +0.21 en todos los
rezagos y la sequía sube a ~+0.14 con 5–8 semanas, sin pasar la prueba
del azar.

In [ ]:
# Rezagos que se revisan: 0 a 8 semanas (o 0 a 2 meses)
REZAGOS = range(0, 9) if GRANULARIDAD == "semanal" else range(0, 3)

# .shift(k) recorre la columna k periodos hacia abajo: en la fila de la semana t
# queda el valor de la semana t-k. Así se compara cada semana con k semanas antes.
rezagos = pd.DataFrame({
    NOMBRE[v]: [sin_epoca[Y].corr(sin_epoca[v].shift(k), method="spearman") for k in REZAGOS]
    for v in VARIABLES
}, index=REZAGOS)
display(rezagos.round(2))

ax = rezagos.plot(marker="o", figsize=(9, 4))
ax.axhspan(-0.1, 0.1, color="#f0efec")  # zona en la que la correlación no dice nada
ax.axhline(0, color="#52514e", lw=1)
ax.set_xlabel(f"rezago ({'semanas' if GRANULARIDAD == 'semanal' else 'meses'} antes)")
ax.set_ylabel("correlación con los casos (sin la época)")
ax.set_title("Clima y aire de los periodos anteriores vs. casos de IRA", loc="left")
ax.legend(frameon=False, ncol=3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"corr_rezagos_{GRANULARIDAD}.png", dpi=150)
plt.show()

if GRANULARIDAD == "semanal":
    # ¿La temperatura de 5 semanas antes podría ser casualidad? (misma prueba por bloques)
    p5 = p_por_bloques(sin_epoca[Y], sin_epoca[TMAX].shift(5))
    print(f"Temperatura 5 semanas antes: r = {rezagos.loc[5, NOMBRE[TMAX]]:+.2f}, p = {p5:.2f}")

## 4. ¿Qué variables de clima y aire se repiten entre sí?

Si dos variables están muy correlacionadas entre sí, cuentan casi la misma
historia. Saberlo sirve para **3.5 (reducción de dimensionalidad)**, que justo
busca resumir variables que se repiten, y para un modelo futuro, donde meter
dos variables casi iguales confunde los resultados.

**Resultado:** con los datos originales, solo la humedad y el ozono se repiten (−0.58). **Sin la época** aparecen más: la temperatura y la humedad
(−0.69), la temperatura y el PM10 (+0.54) y la humedad y el PM10 (−0.55). Es
decir, las semanas más calurosas de lo normal suelen ser más secas y con más
polvo. La sequía casi no se relaciona con las demás.

In [ ]:
# Mapa de calor solo de las variables de clima y aire, sin la época
corr_exp = sin_epoca[VARIABLES].corr(method="spearman").rename(index=NOMBRE, columns=NOMBRE)
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr_exp, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=ax)
ax.set_title("Clima y aire entre sí (sin la época)", loc="left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"corr_exposiciones_{GRANULARIDAD}.png", dpi=150)
plt.show()


def parejas_fuertes(corr: pd.DataFrame, umbral: float = 0.45) -> pd.Series:
    # np.triu(..., k=1) se queda con la mitad de arriba de la matriz (sin la diagonal),
    # para no contar cada pareja dos veces ni la de cada variable consigo misma.
    mitad = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    parejas = mitad.stack()  # una fila por pareja
    return parejas[parejas.abs() >= umbral].sort_values(key=abs, ascending=False).round(2)


corr_exp_originales = base[VARIABLES].corr(method="spearman").rename(index=NOMBRE, columns=NOMBRE)
print("Parejas que se repiten (|r| >= 0.45), datos originales:\n", parejas_fuertes(corr_exp_originales).to_string())
print("\nParejas que se repiten (|r| >= 0.45), sin la época:\n", parejas_fuertes(corr_exp).to_string())

## Conclusiones

Resultados del **panel semanal** (260 semanas, 2015–2019):

1. **Con los datos originales, todo parece relacionado con los casos**, pero es la época del
   año: la temperatura (−0.78) y el PM10 (+0.49) tienen el mismo calendario que
   los casos (invierno frío, con polvo y con muchas infecciones).
2. **Quitando la época, las correlaciones caen a casi 0.** La temperatura pasa de
   −0.78 a −0.02 y el PM10 de +0.49 a +0.12 (*p* = 0.18). Con estos datos no se
   puede decir que el clima o el aire de la misma semana cambien los casos.
3. **Quedan dos señales débiles, para revisar con cuidado:**
   - **Ozono** (+0.21, *p* = 0.03): un poco más de casos en semanas con más
     ozono de lo normal. Es frágil: sin 2016 baja a +0.12, y en 3.3 la
     comparación por grupos no pasó la prueba del azar.
   - **Frío de semanas anteriores** (−0.15 con 5 semanas, *p* ≈ 0.05): después
     de semanas más frías de lo normal hay un poco más casos. Coincide con 3.3.
4. **Varias variables se repiten entre sí** sin la época: temperatura, humedad y
   PM10 (correlaciones de 0.5 a 0.7). Para un modelo, meterlas todas juntas
   confundiría los resultados.

**Panel mensual** (cambiar `GRANULARIDAD = "mensual"`): sale lo mismo. Con los datos originales,
temperatura −0.87 y PM10 +0.54; sin la época, todo cerca de 0 salvo el ozono
(+0.37 con solo 49 meses).

**Qué llevar a 3.5:**

- Usar las variables **sin la época** (comparadas con lo normal para su mes); si
  no, la reducción de dimensionalidad solo va a encontrar "invierno contra
  verano".
- Temperatura, humedad y PM10 se repiten: es esperable que se junten en un solo
  componente.
- Dejar fuera 2020 y revisar el ozono con y sin 2016.

## Consulta con el equipo (checkpoint 2)

Registrar aquí lo que se presentó al equipo sobre las correlaciones encontradas y qué se decidió
(ver `docs/docs/protocolo_eda.md`, sección 5.3).

| Fecha | Participantes | Hallazgo o pregunta presentada | Decisión |
|-------|---------------|-------------------------------|----------|
| _TODO_ | | | |